## Mars Photogrammetry Preprocessing Pipeline

By Christian Tate, Cornell University; Ithaca, NY

https://github.com/cdt59/MPPP


To install the python stand-alone metashape package, visit this [site](v) and download the .whl for your system, then run the command `!pip install Metashape-<version number>-cp37.cp38.cp39.cp310.cp311-none-win_amd64.whl`

In [8]:
# Import python modules

import numpy as np
import cv2, glob, time, os
import matplotlib.pyplot as plt
from numpy.linalg import inv
from scipy import stats
import time
# from planetaryimage import PDS3Image
# import colour_demosaicing
# from PIL import Image
# import matplotlib.cm as cm
# from scipy import interpolate
# from scipy.spatial.transform import Rotation as R
# import colour_demosaicing
import pandas as pd
from scipy.optimize import curve_fit

import Metashape
np.set_printoptions(suppress=True)

from src2.cmod import *
from src2.Image import Image


# %run src/MPPP.py

%matplotlib inline

directory_input  = 'Z:/Mastcam-Z/agisoft/data/'
# directory_input  = 'C:/Users/cdt59/Desktop/agisoft/data'
# directory_input  = 'C:/Users/cdt59/Desktop/agisoft/data/cal'


In [6]:
doc = Metashape.Document( read_only=1 )

# doc.open(path="C:/Users/cdt59/Desktop/agisoft/m32_site_cal690_v2.psx")
# doc.open(path="Z:/Mastcam-Z/agisoft/jezero/m32_site_cal690_.psx")
# doc.open(path="Z:/Mastcam-Z/agisoft/jezero/m32_site_cal690_.psx")
# doc.open(path="C:/Users/cdt59/Desktop/agisoft/m41_site_cal_v202.psx")
# doc.open(path="C:/Users/cdt59/Desktop/agisoft/m41_site_cal_v4.psx")
# doc.open(path="C:/Users/cdt59/Desktop/agisoft/m41_site_cal_v217_.psx")
# doc.open(path="Z:/Mastcam-Z/agisoft/jezero/m41_site_cal_v217_v2_.psx")
# doc.open(path="Z:/Mastcam-Z/agisoft/jezero/m41_site_cal_v222_.psx")
# doc.open(path="C:/Users/cdt59/Desktop/agisoft/m41_site_cal_v224_.psx")
doc.open(path="Z:/Mastcam-Z/agisoft/jezero/m41_site_cal_v230.psx")

# path_references = "C:/Users/cdt59/Desktop/agisoft/m32_site_cal690_rnav_v2.txt"
# path_references = "C:/Users/cdt59/Desktop/agisoft/positions_refs_00_zcam_rnav.txt"
# path_references = "C:/Users/cdt59/Desktop/agisoft/positions_refs_00_zcam_rnav_v4.txt"
# path_references = "C:/Users/cdt59/Desktop/agisoft/positions_refs_00_zcam_rnav_v217.txt"
# path_references = "Z:/Mastcam-Z/agisoft/jezero/positions_refs_zcam_rnav_v217_v2.txt"
# path_references = "C:/Users/cdt59/Desktop/agisoft/m41_site_cal_v224_refs_v4.txt"

# df_r = pd.read_csv( path_references, skiprows=1 )
# df_r

version_name = f'm41_site_cal_v230_v2_{time.strftime("%Y%m%d_%H%M%S")}'

save_path =  '/plots/'+ version_name + '.xlsx'

# doc.chunk.__dir__()

In [7]:
%%time

# %run src/MPPP.py

N_cams = len( doc.chunk.cameras )

columns = [ 'name', 'zcam', 'cam', 'fl', 
            'temp_DEA', 'temp_FPA', 
            'temp_HTR1', 'temp_HTR2', 
            'fmc', 'zmc', 'filtmc', 
            'az_RSM', 'el_RSM', 
            'c_r','a_r','h_r','v_r','o_r','r_r',
            'c_r_est','a_r_est','h_r_est','v_r_est','o_r_est','r_r_est',
            'f_est','cx_est','cy_est','b1_est','b2_est',
            'k0_est','k1_est','k2_est','k3_est','k4_est','p1_est','p2_est',
            'f','cx','cy','b1','b2',
            'k0', 'k1','k2','k3','k4','p1','p2',
            'w','h',
            'X_ref','Y_ref','Z_ref','X_est','Y_est','Z_est',
            'KnKR_e_c_est',
            'q_rpm', 'R_rpm', 't_rpm']
ls = []

for i in range( N_cams )[::-1]:
    if doc.chunk.cameras[ i ].enabled and doc.chunk.cameras[ i ].label[:1] == 'Z' and doc.chunk.cameras[ i ].label[45:48] == '034':
        cam = doc.chunk.cameras[ i ]
        IMG_path = glob.glob( directory_input + '/zcam/*/' + cam.label[:54] + '.IMG' ) + glob.glob( directory_input + '/zcam/' + cam.label[:54] + '.IMG' )
        
        if len(IMG_path) and cam.transform and cam.reference.enabled:
            print( i, cam)
            
            cam = doc.chunk.cameras[i]
            cam_label = cam.label

            zcam = 0 if cam_label[1] == "L" else 1
            cam_name = cam_label[:2] + cam_label[45:48]
            fl = int(cam_label[45:48])

            # transformation between chunk and world coordinate system
            T_chunk2world = np.array(doc.chunk.transform.matrix).reshape(4,4)
            R_chunk2world = np.array(doc.chunk.transform.rotation).reshape(3,3)
            t_chunk2world = np.array(doc.chunk.transform.translation)

            # Camera transformations
            T_cam2chunk = np.array(cam.transform).reshape(4,4)
            R_cam2chunk = T_cam2chunk[:3,:3]
            t_cam2chunk = T_cam2chunk[:3, 3]

            # reconstruction frame to site
            R_world2rover = np.array([[0,1,0], [1,0,0], [0,0,-1]])
            t_world2rover = np.array([0,0,0])
            T_world2rover = np.array([[0,1,0,0],[1,0,0,0],[0,0,-1,0],[0,0,0,1]])


            # h, v & a vectors in agisoft camera frame
            fl = cam.calibration.f
            cx, cy = cam.calibration.cx, cam.calibration.cy
            b1, b2 = cam.calibration.b1, cam.calibration.b2
            hva_cam = np.array([[fl+b1, 0, 0],
                            [b2, fl, 0],
                            [cx, cy, 1]])

            # transform from agisoft camera frame to rover frame
            c_r_est = R_world2rover @ (R_chunk2world @ t_cam2chunk + t_chunk2world)
            hva_r_est = R_world2rover @ R_chunk2world @ R_cam2chunk @ hva_cam
            h_r_est, v_r_est, a_r_est  = hva_r_est[:,0], hva_r_est[:,1], hva_r_est[:,2]
            o_r_est = a_r_est.copy()
            k1, k2 = cam.calibration.k1, cam.calibration.k2
            r_r_est = np.array([0, k1, k2])

            # parse cahvor from PDS label
            IMG_path = glob.glob( directory_input + '/zcam/*/' + cam.label[:54] + '.IMG' ) +\
                        glob.glob( directory_input + '/zcam/' + cam.label[:54] + '.IMG' )
            im = Image(IMG_path=IMG_path[0])
            c_r = np.array(im.label['GEOMETRIC_CAMERA_MODEL']['MODEL_COMPONENT_1'])
            a_r = np.array(im.label['GEOMETRIC_CAMERA_MODEL']['MODEL_COMPONENT_2'])
            h_r = np.array(im.label['GEOMETRIC_CAMERA_MODEL']['MODEL_COMPONENT_3'])
            v_r = np.array(im.label['GEOMETRIC_CAMERA_MODEL']['MODEL_COMPONENT_4'])
            o_r = np.array(im.label['GEOMETRIC_CAMERA_MODEL']['MODEL_COMPONENT_5'])
            r_r = np.array(im.label['GEOMETRIC_CAMERA_MODEL']['MODEL_COMPONENT_6'])

            # Extrinsic and intrinsic camera parameters from PDS labels
            x, y, z = c_r[0], c_r[1], c_r[2]

            hs = np.linalg.norm(np.cross(a_r, h_r), ord=2)
            vs = np.linalg.norm(np.cross(a_r, v_r), ord=2)
            hc = np.dot(h_r, a_r)
            vc = np.dot(v_r, a_r)

            hp = (h_r-hc*a_r)/hs
            vp = (v_r-vc*a_r)/vs
            norm_hp = np.linalg.norm(hp, ord=2)
            norm_vp = np.linalg.norm(vp, ord=2)

            sin_theta = np.clip(np.linalg.norm( np.cross(vp, hp)), a_min=-1, a_max=1)
            cos_theta = np.sqrt(1-sin_theta**2)

            fl = vs
            b1 = - (hs * (-sin_theta)) - vs
            b2 = hs * cos_theta
            cx = hc
            cy = vc
            k0, k1, k2 = r_r[0], r_r[1], r_r[2]

            # random things for logging
            KnKR_est_ = R_chunk2world @ R_world2rover
            t_est = t_chunk2world
            t_ref = np.array(cam.reference.location)
            q_RM = q_wxyz2xyzw( im.label['GEOMETRIC_CAMERA_MODEL']['MODEL_TRANSFORM_QUATERNION'] )
            R_RM = R.from_quat( q_RM ).as_matrix()
            t_RM = im.label['GEOMETRIC_CAMERA_MODEL']['MODEL_TRANSFORM_VECTOR']

            l = [cam.label, zcam, cam_name, int(cam_label[45:48]),
                im.label['INSTRUMENT_STATE_PARMS']['INSTRUMENT_TEMPERATURE'][0], im.label['INSTRUMENT_STATE_PARMS']['INSTRUMENT_TEMPERATURE'][1], 
                im.label['INSTRUMENT_STATE_PARMS']['INSTRUMENT_TEMPERATURE'][2], im.label['INSTRUMENT_STATE_PARMS']['INSTRUMENT_TEMPERATURE'][3], 
                im.focus_mc, im.zoom_mc, im.filter_mc, 
                im.label['RSM_ARTICULATION_STATE']['ARTICULATION_DEVICE_ANGLE'][0], im.label['RSM_ARTICULATION_STATE']['ARTICULATION_DEVICE_ANGLE'][1],
                list(c_r), list(a_r), list(h_r), list(v_r), list(o_r), list(r_r),
                list(c_r_est), list(a_r_est), list(h_r_est), list(v_r_est), list(o_r_est), list(r_r_est),
                cam.calibration.f, cam.calibration.cx, cam.calibration.cy, cam.calibration.b1, cam.calibration.b2,\
                0, cam.calibration.k1, cam.calibration.k2, cam.calibration.k3, cam.calibration.k4, cam.calibration.p1, cam.calibration.p2, 
                fl, cx, cy, b1, b2, 
                k0, k1, k2, 0, 0, 0, 0, 
                cam.calibration.width,cam.calibration.height,
                t_ref[0], t_ref[1], t_ref[2], t_est[0], t_est[1], t_est[2],
                KnKR_est_.tolist(), list(q_RM), R_RM.tolist(), list(t_RM)
                ]

            ls.append( l )

df = pd.DataFrame(ls, columns = columns )

df.to_excel('plots/'+ version_name + '.xlsx')

Number of cameras:43757
41888 <Camera 'ZR0_0965_0752605566_081RAD_N0470000ZCAM05086_0340LMA01'>
41887 <Camera 'ZR0_0965_0752605543_081RAD_N0470000ZCAM05086_0340LMA01'>
41886 <Camera 'ZR0_0965_0752605520_081RAD_N0470000ZCAM05086_0340LMA01'>
41885 <Camera 'ZR0_0965_0752605494_081RAD_N0470000ZCAM05086_0340LMA01'>
41884 <Camera 'ZR0_0965_0752605467_081RAD_N0470000ZCAM05086_0340LMA01'>
41883 <Camera 'ZR0_0965_0752605443_081RAD_N0470000ZCAM05086_0340LMA01'>
41882 <Camera 'ZR0_0965_0752605412_081RAD_N0470000ZCAM05086_0340LMA01'>
41881 <Camera 'ZR0_0965_0752605385_081RAD_N0470000ZCAM05086_0340LMA01'>
41880 <Camera 'ZR0_0965_0752605363_081RAD_N0470000ZCAM05086_0340LMA01'>
41879 <Camera 'ZR0_0965_0752605344_081RAD_N0470000ZCAM05086_0340LMA01'>
41878 <Camera 'ZR0_0965_0752605316_081RAD_N0470000ZCAM05086_0340LMA01'>
41877 <Camera 'ZR0_0965_0752605299_081RAD_N0470000ZCAM05086_0340LMA01'>
41876 <Camera 'ZR0_0965_0752605276_081RAD_N0470000ZCAM05086_0340LMA01'>
41875 <Camera 'ZR0_0965_0752605257_081RA

In [46]:
i = 89

cam = doc.chunk.cameras[i]
cam_label = cam.label

zcam = 0 if cam_label[1] == "L" else 1
cam_name = cam_label[:2] + cam_label[45:48]
fl = int(cam_label[45:48])
print(f"zcam: {zcam}, cam: {cam_name}, fl:{fl}")

# transformation between chunk and world coordinate system
T_chunk2world = np.array(doc.chunk.transform.matrix).reshape(4,4)
R_chunk2world = np.array(doc.chunk.transform.rotation).reshape(3,3)
t_chunk2world = np.array(doc.chunk.transform.translation)
print(f"R_chunk2world:\n{R_chunk2world}")
print(f"t_chunk2world:\n{t_chunk2world}")
print("="*100)

# Camera transformations
T_cam2chunk = np.array(cam.transform).reshape(4,4)
R_cam2chunk = T_cam2chunk[:3,:3]
t_cam2chunk = T_cam2chunk[:3, 3]
print(f"R_cam2chunk:\n{R_cam2chunk}")
print(f"t_cam2chunk:\n{t_cam2chunk}")
print("="*100)

# reconstruction frame to site
R_world2rover = np.array([[0,1,0], [1,0,0], [0,0,-1]])
t_world2rover = np.array([0,0,0])
T_world2rover = np.array([[0,1,0,0],[1,0,0,0],[0,0,-1,0],[0,0,0,1]])
print(f"R_world2rover:\n{R_world2rover}")
print(f"t_world2rover:\n{t_world2rover}")
print("="*100)

# calculate rotation camera to rover agisoft
R_rc_agi = R_world2rover @ R_chunk2world @ R_cam2chunk 
opk_rc_agi = R.from_matrix(R_rc_agi).as_euler('XYZ', degrees=1)

# h, v & a vectors in agisoft camera frame
fl_est = cam.calibration.f
cx_est, cy_est = cam.calibration.cx, cam.calibration.cy
b1_est, b2_est = cam.calibration.b1, cam.calibration.b2
hva_cam_agi = np.array([[fl_est+b1_est, 0, 0],
                    [b2_est, fl_est, 0],
                    [cx_est, cy_est, 1]])

print(f"fl_est:{fl_est:.5f}; b1_est:{b1_est:.5f}; b2_est:{b2_est:.5f}; cx_est:{cx_est:.5f}; cy_est:{cy_est:.5f}")

# transform from agisoft camera frame to rover frame
c_r_est = R_world2rover @ (R_chunk2world @ t_cam2chunk + t_chunk2world)
hva_r_est = R_world2rover @ R_chunk2world @ R_cam2chunk @ hva_cam_agi
h_r_est, v_r_est, a_r_est  = hva_r_est[:,0], hva_r_est[:,1], hva_r_est[:,2]
o_r_est = a_r_est.copy()
k1, k2 = cam.calibration.k1, cam.calibration.k2
r_r_est = np.array([0, k1, k2])

print(f"c_r_est({np.linalg.norm(c_r_est):.5f}): {c_r_est}")
print(f"a_r_est({np.linalg.norm(a_r_est):.5f}): {a_r_est}")
print(f"h_r_est({np.linalg.norm(h_r_est):.5f}): {h_r_est}")
print(f"v_r_est({np.linalg.norm(v_r_est):.5f}): {v_r_est}")
print(f"o_r_est({np.linalg.norm(o_r_est):.5f}): {o_r_est}")
print(f"r_r_est({np.linalg.norm(r_r_est):.5f}): {r_r_est}")
print("="*100)

# parse cahvor from PDS label
IMG_path = glob.glob( directory_input + '/zcam/*/' + cam.label[:54] + '.IMG' ) +\
            glob.glob( directory_input + '/zcam/' + cam.label[:54] + '.IMG' )
im = Image(IMG_path=IMG_path[0])
c_r = np.array(im.label['GEOMETRIC_CAMERA_MODEL']['MODEL_COMPONENT_1'])
a_r = np.array(im.label['GEOMETRIC_CAMERA_MODEL']['MODEL_COMPONENT_2'])
h_r = np.array(im.label['GEOMETRIC_CAMERA_MODEL']['MODEL_COMPONENT_3'])
v_r = np.array(im.label['GEOMETRIC_CAMERA_MODEL']['MODEL_COMPONENT_4'])
o_r = np.array(im.label['GEOMETRIC_CAMERA_MODEL']['MODEL_COMPONENT_5'])
r_r = np.array(im.label['GEOMETRIC_CAMERA_MODEL']['MODEL_COMPONENT_6'])

print(f"c_r({np.linalg.norm(c_r):.5f}): {c_r}")
print(f"a_r({np.linalg.norm(a_r):.5f}): {a_r}")
print(f"h_r({np.linalg.norm(h_r):.5f}): {h_r}")
print(f"v_r({np.linalg.norm(v_r):.5f}): {v_r}")
print(f"o_r({np.linalg.norm(o_r):.5f}): {o_r}")
print(f"r_r({np.linalg.norm(r_r):.5f}): {c_r}")
print("="*100)

# Extrinsic and intrinsic camera parameters from PDS labels
x, y, z = c_r[0], c_r[1], c_r[2]

hs = np.linalg.norm(np.cross(a_r, h_r), ord=2)
vs = np.linalg.norm(np.cross(a_r, v_r), ord=2)
hc = np.dot(h_r, a_r)
vc = np.dot(v_r, a_r)

hp = (h_r-hc*a_r)/hs
vp = (v_r-vc*a_r)/vs
norm_hp = np.linalg.norm(hp, ord=2)
norm_vp = np.linalg.norm(vp, ord=2)

sin_theta = np.clip(np.linalg.norm( np.cross(vp, hp)), a_min=-1, a_max=1)
cos_theta = np.sqrt(1-sin_theta**2)
print(f"sin_theta:{sin_theta}")
print(f"cos_theta:{cos_theta}")

fl = vs
b1 = - (hs * (-sin_theta)) - vs
b2 = hs * cos_theta
cx = hc
cy = vc
k0, k1, k2 = r_r[0], r_r[1], r_r[2]

print(f"fl:{fl:.5f}; b1:{b1:.5f}; b2:{b2:.5f}; cx:{cx:.5f}; cy:{cy:.5f}")
print(f"k0:{k0:.5f}; k1:{k1:.5f}; k2:{k2:.5f}")

# calculate rotation camera to rover from PDS label
Hr = np.array([h_r, v_r, a_r]).T
Hc_pds = np.array([[fl+b1, 0, 0],
            [b2, fl, 0],
            [cx, cy, 1]])
R_rc_pds = Hr @ np.linalg.inv(Hc_pds)
opk_rc_pds = R.from_matrix(R_rc_agi).as_euler('XYZ', degrees=1)


# random things for logging
KnKR_est_ = R_chunk2world @ R_world2rover
t_est = t_chunk2world
t_ref = np.array(cam.reference.location)
q_RM = q_wxyz2xyzw( im.label['GEOMETRIC_CAMERA_MODEL']['MODEL_TRANSFORM_QUATERNION'] )
R_RM = R.from_quat( q_RM ).as_matrix()
t_RM = im.label['GEOMETRIC_CAMERA_MODEL']['MODEL_TRANSFORM_VECTOR']

l = [cam.label, zcam, cam_name, int(cam_label[45:48]),
    im.label['INSTRUMENT_STATE_PARMS']['INSTRUMENT_TEMPERATURE'][0], im.label['INSTRUMENT_STATE_PARMS']['INSTRUMENT_TEMPERATURE'][1], 
    im.label['INSTRUMENT_STATE_PARMS']['INSTRUMENT_TEMPERATURE'][2], im.label['INSTRUMENT_STATE_PARMS']['INSTRUMENT_TEMPERATURE'][3], 
    im.focus_mc, im.zoom_mc, im.filter_mc, 
    im.label['RSM_ARTICULATION_STATE']['ARTICULATION_DEVICE_ANGLE'][0], im.label['RSM_ARTICULATION_STATE']['ARTICULATION_DEVICE_ANGLE'][1],
    c_r, a_r, h_r, v_r, o_r, r_r,
    c_r_est, a_r_est, h_r_est, v_r_est, o_r_est, r_r_est,
    cam.calibration.f, cam.calibration.cx, cam.calibration.cy, cam.calibration.b1, cam.calibration.b2,\
    0, cam.calibration.k1, cam.calibration.k2, cam.calibration.k3, cam.calibration.k4, cam.calibration.p1, cam.calibration.p2, 
    fl, cx, cy, b1, b2, 
    k0, k1, k2, 0, 0, 0, 0, 
    cam.calibration.width,cam.calibration.height,
    t_ref[0], t_ref[1], t_ref[2], t_est[0], t_est[1], t_est[2],
    KnKR_est_, q_RM, R_RM, t_RM
    ]

print('-'*100)
print("|Hc_agi|", np.linalg.norm(hva_cam_agi, axis=0))
print("|Hc_pds|", np.linalg.norm(Hc_pds, axis=0))
print("|Hc_agi|-|Hc_pds|", np.linalg.norm(hva_cam_agi, axis=0) - np.linalg.norm(Hc_pds, axis=0))

print('-'*100)
Hr_est = np.array([h_r_est, v_r_est, a_r_est]).T
Hr_pds = np.array([h_r, v_r, a_r]).T
print("|Hr_est|", np.linalg.norm(Hr_est, axis=0))
print("|Hr_pds|", np.linalg.norm(Hr_pds, axis=0))
print("|Hr_est|-|Hr_pds|", np.linalg.norm(Hr_est, axis=0)-np.linalg.norm(Hr_pds, axis=0))
print('-'*100)


print(f"OPK_rc_agi:{opk_rc_agi}")
print(f"OPK_rc_pds:{opk_rc_pds}")

zcam: 0, cam: ZL110, fl:110
R_chunk2world:
[[ 0.13932423 -0.49803725  0.85588998]
 [ 0.99024136  0.06720102 -0.12209046]
 [ 0.00328891  0.86454782  0.50253981]]
t_chunk2world:
[0.38864466 0.61392766 1.9028849 ]
R_cam2chunk:
[[ 0.11218187 -0.71108273  0.69410128]
 [-0.48666278 -0.64831988 -0.58552598]
 [ 0.86635707 -0.27210786 -0.41878723]]
t_cam2chunk:
[0.61767694 0.0893606  0.09072915]
R_world2rover:
[[ 0  1  0]
 [ 1  0  0]
 [ 0  0 -1]]
t_world2rover:
[0 0 0]
fl_est:14841.16516; b1_est:0.00000; b2_est:0.00000; cx_est:19.21333; cy_est:-7.88257
c_r_est(2.42062): [ 1.22050487  0.50785128 -2.02776791]
a_r_est(1.00000): [0.69910977 0.02988308 0.71438962]
h_r_est(14841.17760): [ -393.08278725 14834.49945199  -208.96039793]
v_r_est(14841.16726): [-10609.36760108   -134.96356296  10377.05880496]
o_r_est(1.00000): [0.69910977 0.02988308 0.71438962]
r_r_est(6.90604): [ 0.          0.64669545 -6.87569594]
c_r(2.17964): [ 0.870362  0.436857 -1.94999 ]
a_r(1.00001): [0.705511  0.0286461 0.708131 ]

In [62]:
Hc_agi = R_cam2chunk.T @ R_chunk2world.T @ R_world2rover.T @ np.array([h_r, v_r, a_r]).T
print(f"H_(c,agi)_pds:\n{Hc_agi}")
print(f"|H_(c,agi)_pds|: {np.linalg.norm(Hc_agi, axis=0)}")
print('-'*75)

print(f"H_(c,agi)_agi:\n{hva_cam_agi}")
print(f"|H_(c,agi)_agi|: {np.linalg.norm(hva_cam_agi, axis=0)}")
print('-'*75)

hva_cam_agi2 = hva_cam_agi + np.array([[0,0,0],[0,0,0],[824, 600, 0]]) 
print(f"H_(c,agi)_agi2:\n{hva_cam_agi2}")
print(f"|H_(c,agi)_agi2|: {np.linalg.norm(hva_cam_agi2, axis=0)}")

H_(c,agi)_pds:
[[14785.17145187    39.83786337    -0.0013178 ]
 [  -37.06591849 14790.73160436    -0.00894083]
 [  796.59174867   587.4810952      0.9999671 ]]
|H_(c,agi)_pds|: [14806.66158044 14802.44785449     1.00000794]
---------------------------------------------------------------------------
H_(c,agi)_agi:
[[14841.16516197     0.             0.        ]
 [    0.         14841.16516197     0.        ]
 [   19.21332528    -7.88256799     1.        ]]
|H_(c,agi)_agi|: [14841.17759872 14841.1672553      1.        ]
---------------------------------------------------------------------------
H_(c,agi)_agi2:
[[14841.16516197     0.             0.        ]
 [    0.         14841.16516197     0.        ]
 [  843.21332528   592.11743201     1.        ]]
|H_(c,agi)_agi2|: [14865.09980043 14852.97230921     1.        ]


In [33]:
i = 89

cam = doc.chunk.cameras[i]

# chunk transformations

# camera transformations
print(f"cam.transform:\n{cam.transform}")
print(f"cam.center:\n{cam.center}")
print(f"cam.Reference.location: {cam.reference.location}")
print(f"cam.Reference.rotation: {cam.reference.rotation}")


cam.transform:
Matrix([[0.11218186537616037, -0.7110827270378504, 0.6941012781929876, 0.6176769377891257],
       [-0.4866627774137288, -0.6483198811185228, -0.5855259796340551, 0.0893606035159949],
       [0.866357068511985, -0.27210785923026654, -0.41878722853549505, 0.09072915028466483],
       [0.0, 0.0, 0.0, 1.0]])
cam.center:
Vector([0.6176769377891257, 0.0893606035159949, 0.09072915028466483])
cam.Reference.location: Vector([0.43703, 0.86503, 1.95395])
cam.Reference.rotation: Vector([44.24486, -1.90631, -0.52153])


In [47]:
cam.calibration.width

1648

In [41]:
doc.chunk.world_crs.

<CoordinateSystem 'Local Coordinates (m)'>

In [22]:

idxs = []
for i in range(len(doc.chunk.cameras)):
    if doc.chunk.cameras[ i ].enabled and doc.chunk.cameras[ i ].label[:1] == 'Z' and doc.chunk.cameras[ i ].label[45:48] == '034':
        cam = doc.chunk.cameras[ i ]
        IMG_path = glob.glob( directory_input + '/zcam/*/' + cam.label[:54] + '.IMG' ) + glob.glob( directory_input + '/zcam/' + cam.label[:54] + '.IMG' )
        if len(IMG_path) and cam.transform and cam.reference.enabled:
            idxs.append(i)

print(len(idxs))


KeyboardInterrupt: 

In [95]:
import pandas as pd

ref_df = pd.read_csv("m41_site_cal_v230_refs_v1.txt", sep="\t")
ref_df  = ref_df[["#Label", "Omega", "Phi", "Kappa", "Omega_est", "Phi_est", "Kappa_est"]].dropna(axis=0, how='any').reset_index(drop=True)

In [96]:
for i in range(ref_df.shape[0]):
    label = ref_df["#Label"][i].split('.')[0]

    found = False
    for cam in doc.chunk.cameras:
        if cam.label == label:
            found = True
            break

    if found and cam.enabled:
        print(i)
        break

0


In [97]:
i = 0

# find camera
label = ref_df["#Label"][i].split('.')[0]

found = False
for cam in doc.chunk.cameras:
    if cam.label == label:
        found = True
        break

print(f"Found Camera: {found}")

# transformation between chunk and world coordinate system
T_chunk2world = np.array(doc.chunk.transform.matrix).reshape(4,4)
R_chunk2world = np.array(doc.chunk.transform.rotation).reshape(3,3)
t_chunk2world = np.array(doc.chunk.transform.translation)

# Camera transformations
T_cam2chunk = np.array(cam.transform).reshape(4,4)
R_cam2chunk = T_cam2chunk[:3,:3]
t_cam2chunk = T_cam2chunk[:3, 3]

# reconstruction frame to site/rover 
T_world2rover = np.array([[0,1,0,0],[1,0,0,0],[0,0,-1,0],[0,0,0,1]])
R_world2rover = np.array([[0,1,0], [1,0,0], [0,0,-1]])
t_world2rover = np.array([0,0,0])

# agisoft camera to site/rover
R_cam2site = R_world2rover @ R_chunk2world @ R_cam2chunk

# convert to opk
R.from_matrix(R_cam2site).as_euler('XYZ', degrees=1)


Found Camera: True


array([ 4.07133289, 43.12773357, 82.68106935])

In [100]:
ref_df[["#Label", "Omega", "Phi", "Kappa", "Omega_est", "Phi_est", "Kappa_est"]].iloc[i,:]

#Label       ZL0_0030_0669603988_492RAD_N0030828ZCAM03103_1...
Omega                                                 43.07667
Phi                                                    2.84448
Kappa                                                  4.49647
Omega_est                                            43.199971
Phi_est                                               2.970217
Kappa_est                                              4.53316
Name: 0, dtype: object